# 6.18 — Vanishing & Exploding Gradients

Vanishing and exploding gradients happen because backpropagation through a deep network multiplies many local derivatives. If most layerwise gains are below 1, the learning signal shrinks toward zero before it reaches early layers; if most are above 1, it can blow up into unstable updates. In this lesson, you will build the arithmetic from scratch in NumPy, inspect the products directly, and see why initialization, activation choice, normalization, and step size are all scale-control tools.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build vanishing and exploding gradients one idea at a time. Run each cell in order and read the printed intermediate values — every product, update, and scale check is visible. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, vector products, and reproducible numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for small random demonstrations.

### 1. A tiny layer: affine signal, gate, and local derivative

A neural layer first forms an affine signal $z=w^\top x+b$, then applies a nonlinearity. The lesson's scratch pass uses $x=[1.5,-0.5]$, weights $[1.6,-0.1]$, and bias $0.300$, so every number can be checked by hand. ReLU keeps positive signals unchanged and kills negative ones; its derivative is therefore either 1 or 0, which means it either passes the gradient backward or blocks it.

In [ ]:
x_w = np.array([1.5, -0.5])          # two input features.
w_w = np.array([1.6, -0.1])          # two weights feeding one neuron.
b_w = 0.300                          # scalar bias.
z_w = float(w_w @ x_w + b_w)         # affine signal before the gate.
h_w = max(0.0, z_w)                  # ReLU activation.
relu_prime_w = 1.0 if z_w > 0 else 0.0  # local derivative of ReLU at z.
print("affine z:", round(z_w, 3))
print("ReLU h:", round(h_w, 3), " ReLU derivative:", relu_prime_w)
assert round(z_w, 3) == 2.750 and round(h_w, 3) == 2.750

▶ What you'll see: the affine score is 2.750, the ReLU output is also 2.750, and the local derivative is 1 because the unit is active.

In [ ]:
parts_w = np.array([w_w[0] * x_w[0], w_w[1] * x_w[1], b_w])  # visible pieces of z.
plt.figure(figsize=(4.4, 3))
plt.bar(["w0*x0", "w1*x1", "b"], parts_w, color=["teal", "orange", "gray"])
plt.axhline(0, color="black", linewidth=0.7)
plt.title("1: pieces of the affine signal")
plt.ylabel("contribution to z")
plt.show()

▶ What you'll see: the first feature contributes 2.4, the second contributes 0.05, and the bias contributes 0.3, summing to 2.75.

*Why it's done this way:* Backpropagation is local. Each layer only needs its own derivative, but deep learning composes many such local choices. A ReLU derivative of 1 preserves gradient scale at this unit; a derivative of 0 would make the upstream gradient vanish immediately along this path.

### 2. Backpropagation is a product of local gains

For a chain $h_0\to h_1\to\cdots\to h_L\to L$, the chain rule multiplies local derivatives: $\frac{\partial L}{\partial h_0}=\left(\prod_{\ell=1}^L s_\ell\right)\frac{\partial L}{\partial h_L}$. That is the whole mechanism. A single gain near 1 looks harmless, but repeating it many times makes exponential shrinkage or growth.

In [ ]:
gains_w = np.array([0.9, 0.8, 1.1, 0.7, 0.95])  # five layerwise derivative magnitudes.
grad_out_w = 2.0                                 # gradient arriving from the loss side.
product_w = float(np.prod(gains_w))              # chain-rule multiplier.
grad_in_w = product_w * grad_out_w               # gradient reaching the earliest state.
print("local gains:", gains_w)
print("product:", round(product_w, 4), " input gradient:", round(grad_in_w, 4))
assert round(product_w, 4) == 0.5267

▶ What you'll see: the five local gains multiply to about 0.527, so a gradient of 2.0 becomes about 1.053.

In [ ]:
prefix_w = grad_out_w * np.cumprod(gains_w[::-1])[::-1]  # gradient size after crossing suffixes of the chain.
plt.figure(figsize=(4.6, 3))
plt.plot(np.arange(len(prefix_w)), prefix_w, marker="o", color="purple")
plt.title("2: gradient after repeated local products")
plt.xlabel("earlier position in the chain")
plt.ylabel("gradient magnitude")
plt.show()

▶ What you'll see: the gradient changes step by step as each local factor is multiplied into the signal.

*Why it's done this way:* The chain rule is unavoidable for composed functions. The danger is not one bad layer; it is repeated multiplication. Products below 1 erase information geometrically, while products above 1 amplify noise and curvature geometrically.

### 3. Vanishing versus exploding as depth grows

Now hold the local gain constant so the depth effect is isolated. A gain of 0.8 becomes $0.8^L$, while a gain of 1.2 becomes $1.2^L$. At depth 30 those are not slightly different; they live on very different scales.

In [ ]:
depths_w = np.arange(1, 31)              # depths from 1 to 30.
vanish_w = 0.8 ** depths_w               # repeated below-one gain.
explode_w = 1.2 ** depths_w              # repeated above-one gain.
print("0.8^30:", round(float(vanish_w[-1]), 4))
print("1.2^30:", round(float(explode_w[-1]), 3))
assert round(float(vanish_w[-1]), 4) == 0.0012
assert round(float(explode_w[-1]), 3) == 237.376

▶ What you'll see: the vanishing product is about 0.0012, while the exploding product is about 237.376.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(depths_w, vanish_w, label="0.8^depth", color="steelblue")
plt.plot(depths_w, explode_w, label="1.2^depth", color="crimson")
plt.yscale("log")
plt.title("3: products separate exponentially")
plt.xlabel("depth")
plt.ylabel("gradient multiplier, log scale")
plt.legend()
plt.show()

▶ What you'll see: on a log scale, the below-one product slopes downward and the above-one product slopes upward.

*Why it's done this way:* Exponentials are the natural result of repeated multiplication. This is why deep networks care so much about keeping the typical layerwise gain near 1: depth turns small scale errors into training failures.

### 4. Activation derivatives can quietly shrink gradients

Sigmoid is useful as a squashing function, but its derivative is $\sigma(z)(1-\sigma(z))$, which is at most 0.25 and becomes tiny when $z$ is far from 0. ReLU does not saturate on the positive side, but it has derivative 0 on the negative side. Different nonlinearities therefore create different gradient pathways.

In [ ]:
z_grid_w = np.linspace(-6, 6, 301)                      # preactivation values.
sig_w = 1 / (1 + np.exp(-z_grid_w))                     # sigmoid activation.
sig_prime_w = sig_w * (1 - sig_w)                       # sigmoid derivative.
relu_prime_grid_w = (z_grid_w > 0).astype(float)        # ReLU derivative except exactly at 0.
print("max sigmoid derivative:", round(float(sig_prime_w.max()), 3))
print("sigmoid derivative at z=6:", round(float(sig_prime_w[-1]), 4))
assert round(float(sig_prime_w.max()), 3) == 0.25

▶ What you'll see: sigmoid's best possible derivative is 0.25, and it is almost flat at large positive inputs.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(z_grid_w, sig_prime_w, label="sigmoid derivative", color="purple")
plt.plot(z_grid_w, relu_prime_grid_w, label="ReLU derivative", color="teal", alpha=0.8)
plt.title("4: activation derivatives set local gain")
plt.xlabel("preactivation z")
plt.ylabel("local derivative")
plt.legend()
plt.show()

▶ What you'll see: sigmoid has a small bell-shaped derivative, while ReLU passes gradient as 1 on positive inputs and 0 on negative inputs.

*Why it's done this way:* The derivative of the activation is one factor in the chain-rule product. Saturating activations create many below-one factors, and inactive ReLUs create exact zeros; both can starve early layers of gradient.

### 5. Initialization controls signal and gradient variance

Weights also multiply signals and gradients. If each layer's weights are too small, activations and gradients contract; if too large, they expand. A scratch simulation with random matrices shows why scale-aware initialization divides by input dimension: it tries to keep variance from drifting as depth increases.

In [ ]:
rng_w = np.random.default_rng(0)
width_w = 128
layers_w = 30
x0_w = rng_w.normal(size=(width_w, 1))
scales_w = [0.05, np.sqrt(2 / width_w), 0.30]  # too small, He-like for ReLU, too large.
labels_w = ["too small", "He-like", "too large"]
print("He-like scale:", round(scales_w[1], 4))
assert round(scales_w[1], 4) == 0.125

▶ What you'll see: the He-like scale for width 128 is 0.125, much smaller than an arbitrary large weight scale.

In [ ]:
norms_by_scale_w = []
for scale_w in scales_w:
    h_tmp_w = x0_w.copy()
    norms_w = []
    for layer_w in range(layers_w):
        W_tmp_w = rng_w.normal(0, scale_w, size=(width_w, width_w))
        h_tmp_w = np.maximum(0, W_tmp_w @ h_tmp_w)
        norms_w.append(float(np.linalg.norm(h_tmp_w)))
    norms_by_scale_w.append(norms_w)
print("final activation norms:", [round(curve[-1], 3) for curve in norms_by_scale_w])

▶ What you'll see: small weights drive norms toward zero, large weights inflate norms, and scale-aware weights stay more controlled.

In [ ]:
plt.figure(figsize=(5, 3))
for curve_w, label_w in zip(norms_by_scale_w, labels_w):
    plt.plot(curve_w, label=label_w)
plt.yscale("log")
plt.title("5: activation scale through depth")
plt.xlabel("layer")
plt.ylabel("activation norm, log scale")
plt.legend()
plt.show()

▶ What you'll see: the curves separate by orders of magnitude, showing that weight scale is a training-design choice, not a cosmetic detail.

*Why it's done this way:* Variance-preserving initialization is a practical attempt to make the typical product of weight scale and activation derivative close to neutral. It cannot guarantee perfect gradients, but it prevents obvious exponential drift at the starting point.

### 6. Normalization and updates keep arithmetic in a usable range

The content block normalizes the score 2.750 with mean 1.000 and variance 0.250, giving $(2.750-1.000)/\sqrt{0.250+10^{-5}}\approx3.500$. Normalization does not remove the need for gradients, but it gives each layer a more predictable scale. Then the optimizer makes a small parameter move, such as $2.000-0.080\cdot1.800=1.856$.

In [ ]:
score_w = 2.750
mean_w = 1.000
var_w = 0.250
eps_w = 0.00001
normed_w = (score_w - mean_w) / np.sqrt(var_w + eps_w)
theta_w = 2.000
eta_w = 0.080
grad_w = 1.800
theta_new_w = theta_w - eta_w * grad_w
print("normalized score:", round(float(normed_w), 3))
print("parameter update:", round(theta_new_w, 3))
assert round(float(normed_w), 3) == 3.5 and round(theta_new_w, 3) == 1.856

▶ What you'll see: the normalized score is 3.500 and the parameter moves from 2.000 to 1.856.

In [ ]:
batch_scores_w = np.array([0.4, 1.0, 2.75, 4.2])
centered_w = (batch_scores_w - mean_w) / np.sqrt(var_w + eps_w)
plt.figure(figsize=(4.6, 3))
plt.bar(["0.4", "1.0", "2.75", "4.2"], centered_w, color="darkorange")
plt.axhline(0, color="black", linewidth=0.7)
plt.title("6: normalized values are deviations from scale")
plt.ylabel("normalized value")
plt.show()

▶ What you'll see: values below the mean become negative, the mean becomes 0, and high scores become positive deviations.

*Why it's done this way:* Normalization attacks the scale problem before the chain-rule product gets too large or too small. The optimizer then relies on gradients that are numerically meaningful; if the gradient is already vanished or exploded, the update formula itself cannot rescue learning.

### 7. Scores, softmax, and memory bookkeeping

A model often turns raw scores into probabilities, so unstable scores can also create unstable comparisons. With scores 2.750 and 0.400, softmax assigns probability $e^{2.750}/(e^{2.750}+e^{0.400})\approx0.913$. Separately, a tiny activation block with 4 vectors of length 128 in 32-bit floats uses $4\cdot128\cdot4/1024=2$ KB; deep networks repeat this bookkeeping across many layers.

In [ ]:
logits_w = np.array([2.750, 0.400])
exp_w = np.exp(logits_w)
prob_w = exp_w[0] / np.sum(exp_w)
bytes_w = 4 * 128 * 4
kb_w = bytes_w / 1024
print("exp values:", np.round(exp_w, 3))
print("softmax probability for score 2.75:", round(float(prob_w), 3))
print("activation memory KB:", round(kb_w, 3))
assert round(float(prob_w), 3) == 0.913 and round(kb_w, 3) == 2.0

▶ What you'll see: the high score receives probability 0.913, and the small activation block uses 2 KB.

In [ ]:
logit_gap_w = np.linspace(0, 10, 101)
prob_curve_w = 1 / (1 + np.exp(-logit_gap_w))
plt.figure(figsize=(4.8, 3))
plt.plot(logit_gap_w, prob_curve_w, color="teal")
plt.scatter([2.35], [prob_w], color="red", zorder=3)
plt.title("7: softmax confidence grows with logit gap")
plt.xlabel("score gap")
plt.ylabel("probability of larger score")
plt.show()

▶ What you'll see: probability saturates as the score gap grows, so very large logits can make gradients less informative.

*Why it's done this way:* Deep learning arithmetic is not just symbolic. Scale affects probabilities, gradients, and stored activations, so stable training means managing math and hardware together.

## 🛠️ Setup

In [ ]:
import numpy as np # Load NumPy for arrays, dot products, random simulations, and numerical checks.
import matplotlib.pyplot as plt # Load Matplotlib for small diagnostic plots.
np.random.seed(0) # Make stochastic examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Multiply local gains

**Goal.** Compute one chain-rule product, because the earliest gradient is the final gradient times every local derivative in between.

In [ ]:
gains_b1 = np.array([0.9, 0.8, 1.1, 0.7, 0.95]) # Define five local derivative magnitudes.
grad_tail_b1 = 2.0 # Define the gradient arriving from the loss side.
product_b1 = float(np.prod(gains_b1)) # Multiply local gains into one chain-rule scale factor.
grad_head_b1 = grad_tail_b1 * product_b1 # Scale the tail gradient to get the head gradient.
print("product:", round(product_b1, 4), "head gradient:", round(grad_head_b1, 4)) # Inspect the result.
assert round(product_b1, 4) == 0.5267 # Verify the concrete chain-rule product.
plt.figure(figsize=(4, 3)) # Create a compact gain plot.
plt.bar(np.arange(len(gains_b1)), gains_b1, color="teal") # Show each local derivative magnitude.
plt.axhline(1, color="black", linestyle="--") # Mark neutral gain.
plt.title("Basic 1: local gains") # Title the plot.
plt.xlabel("layer") # Label layers.
plt.ylabel("gain") # Label gain size.
plt.show() # Display the figure.

▶ What you'll see: several gains sit below 1, and the product shrinks the gradient from 2.0 to about 1.053.

👀 Takeaway: deep gradients are products, so every local scale factor matters.

### Basic 2 — See a vanishing product

**Goal.** Raise a below-one factor to increasing depths, because repeated multiplication by 0.8 makes gradients disappear exponentially.

In [ ]:
depths_b2 = np.arange(1, 21) # Define depths from 1 through 20.
mult_b2 = 0.8 ** depths_b2 # Compute the gradient multiplier at each depth.
print("0.8^10:", round(float(mult_b2[9]), 4), "0.8^20:", round(float(mult_b2[-1]), 4)) # Inspect two depths.
assert round(float(mult_b2[9]), 4) == 0.1074 # Verify the depth-10 multiplier.
plt.figure(figsize=(4, 3)) # Create a line plot.
plt.plot(depths_b2, mult_b2, marker="o", color="steelblue") # Plot decay with depth.
plt.title("Basic 2: vanishing product") # Title the plot.
plt.xlabel("depth") # Label the depth axis.
plt.ylabel("0.8^depth") # Label the multiplier.
plt.show() # Display the plot.

▶ What you'll see: the multiplier drops quickly, reaching about 0.0115 by depth 20.

👀 Takeaway: a modest below-one gain becomes tiny when depth repeats it.

### Basic 3 — See an exploding product

**Goal.** Raise an above-one factor to increasing depths, because repeated multiplication by 1.2 makes gradients grow exponentially.

In [ ]:
depths_b3 = np.arange(1, 21) # Define depths from 1 through 20.
mult_b3 = 1.2 ** depths_b3 # Compute the exploding multiplier at each depth.
print("1.2^10:", round(float(mult_b3[9]), 3), "1.2^20:", round(float(mult_b3[-1]), 3)) # Inspect two depths.
assert round(float(mult_b3[9]), 3) == 6.192 # Verify the depth-10 multiplier.
plt.figure(figsize=(4, 3)) # Create a line plot.
plt.plot(depths_b3, mult_b3, marker="o", color="crimson") # Plot growth with depth.
plt.title("Basic 3: exploding product") # Title the plot.
plt.xlabel("depth") # Label the depth axis.
plt.ylabel("1.2^depth") # Label the multiplier.
plt.show() # Display the plot.

▶ What you'll see: the multiplier rises from 1.2 to more than 38 by depth 20.

👀 Takeaway: a slightly high gain can make early-layer gradients dangerously large.

### Basic 4 — ReLU gate derivative

**Goal.** Compute ReLU outputs and derivatives, because inactive ReLUs send zero gradient backward.

In [ ]:
z_b4 = np.array([-2.0, -0.1, 0.0, 1.5, 2.75]) # Define representative preactivation values.
h_b4 = np.maximum(0, z_b4) # Apply ReLU to each value.
deriv_b4 = (z_b4 > 0).astype(float) # Compute the ReLU derivative away from zero.
print("ReLU outputs:", h_b4) # Inspect forward activations.
print("ReLU derivatives:", deriv_b4) # Inspect local backward gates.
assert int(np.sum(deriv_b4 == 0)) == 3 # Verify three values block gradients in this convention.
plt.figure(figsize=(4, 3)) # Create a derivative plot.
plt.bar([str(v) for v in z_b4], deriv_b4, color="darkorange") # Draw derivative by preactivation.
plt.title("Basic 4: ReLU backward gate") # Title the plot.
plt.xlabel("z") # Label preactivation values.
plt.ylabel("d ReLU / dz") # Label derivative.
plt.show() # Display the plot.

▶ What you'll see: negative and zero preactivations have derivative 0, while positive preactivations have derivative 1.

👀 Takeaway: ReLU helps positive paths keep scale, but dead paths vanish exactly.

### Basic 5 — Sigmoid derivative shrinks

**Goal.** Compute sigmoid derivatives, because saturation creates tiny local gains.

In [ ]:
z_b5 = np.array([-6.0, -2.0, 0.0, 2.0, 6.0]) # Define preactivations from saturated negative to saturated positive.
sig_b5 = 1 / (1 + np.exp(-z_b5)) # Compute sigmoid values.
deriv_b5 = sig_b5 * (1 - sig_b5) # Compute sigmoid derivatives.
print("sigmoid:", np.round(sig_b5, 3)) # Inspect activations.
print("derivative:", np.round(deriv_b5, 4)) # Inspect local gains.
assert round(float(deriv_b5[2]), 3) == 0.25 # Verify the maximum derivative at zero.
plt.figure(figsize=(4, 3)) # Create a compact bar plot.
plt.bar([str(v) for v in z_b5], deriv_b5, color="purple") # Show derivative size by preactivation.
plt.title("Basic 5: sigmoid local gains") # Title the plot.
plt.xlabel("z") # Label preactivation values.
plt.ylabel("sigmoid'(z)") # Label derivative values.
plt.show() # Display the plot.

▶ What you'll see: the derivative peaks at 0.25 near z=0 and is tiny near ±6.

👀 Takeaway: saturating activations insert many below-one factors into the gradient product.

### Basic 6 — One gradient-descent step

**Goal.** Apply $\theta\leftarrow\theta-\eta g$, because gradient scale matters only through the parameter move it causes.

In [ ]:
theta_b6 = 2.000 # Define the current scalar parameter.
eta_b6 = 0.080 # Define the learning rate.
grad_b6 = 1.800 # Define the scalar gradient.
step_b6 = eta_b6 * grad_b6 # Compute the amount subtracted from the parameter.
theta_new_b6 = theta_b6 - step_b6 # Apply one gradient-descent update.
print("step:", round(step_b6, 3), "new theta:", round(theta_new_b6, 3)) # Inspect the movement.
assert round(theta_new_b6, 3) == 1.856 # Verify the lesson update.
plt.figure(figsize=(4, 3)) # Create a before-after plot.
plt.bar(["before", "after"], [theta_b6, theta_new_b6], color=["gray", "teal"]) # Compare parameter values.
plt.title("Basic 6: one optimizer nudge") # Title the plot.
plt.ylabel("parameter value") # Label the value axis.
plt.show() # Display the plot.

▶ What you'll see: the parameter moves from 2.000 down to 1.856, a small reliable nudge.

👀 Takeaway: vanished gradients cause tiny nudges; exploded gradients cause oversized nudges.

### Basic 7 — Normalize one score

**Goal.** Standardize a signal with mean and variance, because normalization measures deviations on a controlled scale.

In [ ]:
score_b7 = 2.750 # Define the raw score from the scratch pass.
mean_b7 = 1.000 # Define the normalization mean.
var_b7 = 0.250 # Define the normalization variance.
eps_b7 = 0.00001 # Add a small epsilon to avoid division by zero.
normalized_b7 = (score_b7 - mean_b7) / np.sqrt(var_b7 + eps_b7) # Compute the normalized value.
print("normalized value:", round(float(normalized_b7), 3)) # Inspect the standardized score.
assert round(float(normalized_b7), 3) == 3.5 # Verify the lesson value.
plt.figure(figsize=(4, 3)) # Create a compact comparison plot.
plt.bar(["raw", "mean", "normalized"], [score_b7, mean_b7, normalized_b7], color=["orange", "gray", "teal"]) # Compare raw and normalized quantities.
plt.title("Basic 7: normalization arithmetic") # Title the plot.
plt.show() # Display the plot.

▶ What you'll see: the score is 3.5 standard-deviation units above the chosen mean.

👀 Takeaway: normalization makes scale explicit before gradients propagate through depth.

### Basic 8 — Softmax from two scores

**Goal.** Convert two scores into a probability, because losses usually compare logits through exponentials.

In [ ]:
logits_b8 = np.array([2.750, 0.400]) # Define the lesson score and baseline.
exp_b8 = np.exp(logits_b8) # Exponentiate both logits.
prob_b8 = exp_b8[0] / np.sum(exp_b8) # Compute the two-class softmax probability for the first logit.
print("exp(logits):", np.round(exp_b8, 3)) # Inspect exponential scores.
print("probability:", round(float(prob_b8), 3)) # Inspect the normalized comparison.
assert round(float(prob_b8), 3) == 0.913 # Verify the lesson softmax probability.
plt.figure(figsize=(4, 3)) # Create a probability bar chart.
plt.bar(["score 2.75", "score 0.40"], [prob_b8, 1 - prob_b8], color=["teal", "gray"]) # Show both class probabilities.
plt.title("Basic 8: two-score softmax") # Title the plot.
plt.ylabel("probability") # Label probability axis.
plt.show() # Display the plot.

▶ What you'll see: the larger score gets about 91.3% probability.

👀 Takeaway: logits with large gaps can saturate probabilities and weaken useful gradients.

### Basic 9 — Activation memory

**Goal.** Compute activation memory, because backpropagation stores intermediate values and depth multiplies the cost.

In [ ]:
vectors_b9 = 4 # Define how many activation vectors are stored.
length_b9 = 128 # Define the length of each vector.
bytes_per_float_b9 = 4 # Use 32-bit floats.
kb_b9 = vectors_b9 * length_b9 * bytes_per_float_b9 / 1024 # Convert bytes to kilobytes.
print("activation memory KB:", round(kb_b9, 3)) # Inspect memory cost.
assert round(kb_b9, 3) == 2.0 # Verify the lesson memory number.
plt.figure(figsize=(4, 3)) # Create a simple memory plot.
plt.bar(["activations"], [kb_b9], color="slateblue") # Show the memory amount.
plt.title("Basic 9: stored activation memory") # Title the plot.
plt.ylabel("KB") # Label memory axis.
plt.show() # Display the plot.

▶ What you'll see: even a tiny block uses 2 KB; real networks repeat this many times.

👀 Takeaway: stable deep learning is also memory bookkeeping, not just calculus.

### Basic 10 — Compare safe and unsafe update sizes

**Goal.** Compare updates from small, normal, and huge gradients, because gradient scale directly controls parameter movement.

In [ ]:
theta_b10 = 2.0 # Define a starting parameter.
eta_b10 = 0.08 # Define one learning rate.
grads_b10 = np.array([0.001, 1.8, 100.0]) # Define vanished, usable, and exploded gradients.
updates_b10 = eta_b10 * grads_b10 # Compute parameter movements.
new_thetas_b10 = theta_b10 - updates_b10 # Apply the update formula to each gradient.
print("updates:", np.round(updates_b10, 4)) # Inspect movement magnitudes.
print("new thetas:", np.round(new_thetas_b10, 4)) # Inspect resulting parameters.
assert round(float(updates_b10[1]), 3) == 0.144 # Verify the normal update size.
plt.figure(figsize=(4.5, 3)) # Create a comparison plot.
plt.bar(["vanished", "usable", "exploded"], updates_b10, color=["steelblue", "teal", "crimson"]) # Show update magnitudes.
plt.yscale("log") # Use log scale so all three are visible.
plt.title("Basic 10: update scale") # Title the plot.
plt.ylabel("|eta * gradient|, log scale") # Label update size.
plt.show() # Display the plot.

▶ What you'll see: the vanished update is nearly invisible, while the exploded update is orders of magnitude larger.

👀 Takeaway: vanishing and exploding gradients are optimizer problems because they become bad update sizes.

## 🟡 Easy

### Easy 1 — Backpropagate through a scalar deep chain

**Goal.** Build a scalar network $h_L=(a^L)x$ and compute its exact input gradient, because this is the cleanest possible vanishing/exploding demo.

In [ ]:
x_e1 = 1.0 # Define the scalar input.
depth_e1 = 12 # Choose a moderately deep scalar chain.
gains_e1 = np.array([0.85, 1.00, 1.15]) # Compare shrinking, neutral, and growing chains.
outputs_e1 = x_e1 * (gains_e1 ** depth_e1) # Forward output for h=a^L x.
grads_e1 = gains_e1 ** depth_e1 # Backward derivative dh/dx is the same product.
print("outputs:", np.round(outputs_e1, 4)) # Inspect final activations.
print("input gradients:", np.round(grads_e1, 4)) # Inspect exact gradients.
assert round(float(grads_e1[0]), 4) == 0.1422 # Verify the shrinking chain value.
plt.figure(figsize=(4.5, 3)) # Create a compact comparison plot.
plt.bar(["a=0.85", "a=1.00", "a=1.15"], grads_e1, color=["steelblue", "gray", "crimson"]) # Compare gradient products.
plt.title("Easy 1: scalar chain gradients") # Title the plot.
plt.ylabel("dh_L/dx") # Label derivative axis.
plt.show() # Display the plot.

▶ What you'll see: the below-one chain produces a small gradient, the neutral chain stays at 1, and the above-one chain grows.

👀 Takeaway: deep-gradient behavior can be understood from the scalar chain-rule product before adding matrices.

### Easy 2 — Matrix products can change gradient norm

**Goal.** Multiply a gradient by repeated weight matrices, because dense layers backpropagate through transposed weight products.

In [ ]:
rng_e2 = np.random.default_rng(2) # Create reproducible matrices.
width_e2 = 20 # Define vector width.
depth_e2 = 15 # Define number of layers.
scales_e2 = [0.05, 0.22, 0.45] # Compare small, moderate, and large matrix scales.
final_norms_e2 = [] # Store final gradient norms.
for scale_e2 in scales_e2: # Loop through scales.
    g_e2 = np.ones(width_e2) / np.sqrt(width_e2) # Start with unit-norm output gradient.
    for layer_e2 in range(depth_e2): # Backpropagate through random matrices.
        W_e2 = rng_e2.normal(0, scale_e2, size=(width_e2, width_e2)) # Create one random layer matrix.
        g_e2 = W_e2.T @ g_e2 # Apply the transposed matrix to the gradient.
    final_norms_e2.append(float(np.linalg.norm(g_e2))) # Store the final input-gradient norm.
print("final gradient norms:", np.round(final_norms_e2, 6)) # Inspect scale effects.
assert final_norms_e2[0] < final_norms_e2[1] < final_norms_e2[2] # Verify monotonic growth in this seeded demo.
plt.figure(figsize=(4.5, 3)) # Create a scale comparison plot.
plt.bar(["0.05", "0.22", "0.45"], final_norms_e2, color=["steelblue", "teal", "crimson"]) # Show final norms.
plt.yscale("log") # Use log scale for readability.
plt.title("Easy 2: matrix backprop scale") # Title the plot.
plt.xlabel("weight std") # Label scale choices.
plt.ylabel("input-gradient norm") # Label gradient norm.
plt.show() # Display the plot.

▶ What you'll see: small matrices nearly erase the gradient, while large matrices amplify it.

👀 Takeaway: in real dense networks, the chain product is a product of matrices, not just scalars.

### Easy 3 — Initialization sweep for forward variance

**Goal.** Track activation variance through random ReLU layers, because unstable forward scale usually predicts unstable backward scale too.

In [ ]:
rng_e3 = np.random.default_rng(3) # Create reproducible randomness.
width_e3 = 100 # Define layer width.
depth_e3 = 20 # Define number of layers.
x_e3 = rng_e3.normal(size=(width_e3, 200)) # Simulate a batch of 200 examples.
scales_e3 = np.array([0.05, np.sqrt(2 / width_e3), 0.25]) # Compare too small, He-like, and too large.
labels_e3 = ["small", "He-like", "large"] # Name the scale choices.
variance_curves_e3 = [] # Store variance by layer.
for scale_e3 in scales_e3: # Loop over initialization scales.
    h_e3 = x_e3.copy() # Reset the input batch.
    curve_e3 = [] # Store this scale's layer variances.
    for layer_e3 in range(depth_e3): # Run a forward pass through random ReLU layers.
        W_e3 = rng_e3.normal(0, scale_e3, size=(width_e3, width_e3)) # Create weights.
        h_e3 = np.maximum(0, W_e3 @ h_e3) # Apply affine map and ReLU gate.
        curve_e3.append(float(np.var(h_e3))) # Store activation variance.
    variance_curves_e3.append(curve_e3) # Store the curve.
print("final variances:", [round(c[-1], 4) for c in variance_curves_e3]) # Inspect final scale.

▶ What you'll see: the final variances separate strongly across initialization scales.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a variance-through-depth figure.
for curve_e3, label_e3 in zip(variance_curves_e3, labels_e3): # Plot each scale curve.
    plt.plot(curve_e3, label=label_e3) # Draw the layer-by-layer variance.
plt.yscale("log") # Use log scale to show shrinkage and growth.
plt.title("Easy 3: forward variance by initialization") # Title the plot.
plt.xlabel("layer") # Label layer axis.
plt.ylabel("activation variance") # Label variance axis.
plt.legend() # Show scale labels.
plt.show() # Display the plot.

▶ What you'll see: scale-aware initialization slows variance drift compared with obviously small or large weights.

👀 Takeaway: good initialization tries to keep both forward activations and backward gradients in a usable range.

### Easy 4 — Normalization changes effective gradient scale

**Goal.** Normalize a batch and compare gradients through the scale factor, because dividing by standard deviation rescales backward signals.

In [ ]:
scores_e4 = np.array([0.4, 1.0, 2.75, 4.2]) # Define a tiny batch of raw scores.
mean_e4 = float(np.mean(scores_e4)) # Compute batch mean.
var_e4 = float(np.var(scores_e4)) # Compute batch variance.
std_e4 = np.sqrt(var_e4 + 1e-5) # Compute stable standard deviation.
normed_e4 = (scores_e4 - mean_e4) / std_e4 # Normalize scores.
upstream_e4 = np.ones_like(scores_e4) # Use a unit upstream gradient for inspection.
approx_grad_e4 = upstream_e4 / std_e4 # Show the main scaling effect of normalization.
print("mean:", round(mean_e4, 3), "std:", round(float(std_e4), 3)) # Inspect scale statistics.
print("normalized:", np.round(normed_e4, 3)) # Inspect normalized values.
print("approx gradient scale:", round(float(approx_grad_e4[0]), 3)) # Inspect backward rescale.
assert round(float(np.mean(normed_e4)), 6) == 0.0 # Verify normalized mean.
plt.figure(figsize=(4.5, 3)) # Create a normalized-value plot.
plt.bar(np.arange(len(scores_e4)), normed_e4, color="darkorange") # Show normalized scores.
plt.axhline(0, color="black", linewidth=0.7) # Mark zero mean.
plt.title("Easy 4: batch normalization effect") # Title the plot.
plt.ylabel("normalized score") # Label normalized scale.
plt.show() # Display the plot.

▶ What you'll see: normalized scores are centered at zero, and the backward scale is tied to the batch standard deviation.

👀 Takeaway: normalization manages gradient scale by making layer inputs live on a predictable numeric scale.

### Easy 5 — Detect an unsafe gradient norm

**Goal.** Measure gradient norm before an update, because exploding gradients are often diagnosed by unusually large norms.

In [ ]:
grad_e5 = np.array([3.0, -4.0, 12.0]) # Define a three-parameter gradient.
eta_e5 = 0.05 # Define a learning rate.
norm_e5 = float(np.linalg.norm(grad_e5)) # Compute gradient L2 norm.
update_e5 = -eta_e5 * grad_e5 # Compute the parameter update vector.
update_norm_e5 = float(np.linalg.norm(update_e5)) # Compute update norm.
print("gradient norm:", round(norm_e5, 3), "update norm:", round(update_norm_e5, 3)) # Inspect safety diagnostics.
assert round(norm_e5, 3) == 13.0 # Verify the 3-4-12 triangle norm.
plt.figure(figsize=(4, 3)) # Create a component plot.
plt.bar(["g0", "g1", "g2"], grad_e5, color="crimson") # Show gradient components.
plt.axhline(0, color="black", linewidth=0.7) # Mark zero.
plt.title("Easy 5: gradient components") # Title the plot.
plt.ylabel("gradient value") # Label gradient axis.
plt.show() # Display the plot.

▶ What you'll see: the gradient norm is 13, so even a modest learning rate gives a sizeable update norm of 0.65.

👀 Takeaway: norm checks turn "exploding" from a vague word into a measurable training signal.

## 🔴 Advanced

### Advanced 1 — Compare depth and gain on one heatmap

**Goal.** Map gradient multipliers across many gains and depths, because vanishing/exploding is a two-variable scale problem.

In [ ]:
gains_a1 = np.linspace(0.7, 1.3, 61) # Define local gains from shrinking to growing.
depths_a1 = np.arange(1, 41) # Define depths from 1 to 40.
log10_mult_a1 = np.array([[d_a1 * np.log10(g_a1) for g_a1 in gains_a1] for d_a1 in depths_a1]) # Compute log10(gain^depth).
print("log10 multiplier at gain .8 depth 30:", round(float(30 * np.log10(0.8)), 3)) # Inspect a vanishing point.
print("log10 multiplier at gain 1.2 depth 30:", round(float(30 * np.log10(1.2)), 3)) # Inspect an exploding point.
assert round(float(30 * np.log10(0.8)), 3) == -2.907 # Verify the vanishing log scale.
plt.figure(figsize=(5, 3.5)) # Create a heatmap figure.
plt.imshow(log10_mult_a1, aspect="auto", origin="lower", extent=[gains_a1[0], gains_a1[-1], depths_a1[0], depths_a1[-1]], cmap="coolwarm") # Draw log multipliers.
plt.colorbar(label="log10 gradient multiplier") # Add color scale.
plt.axvline(1.0, color="black", linestyle="--") # Mark neutral gain.
plt.title("Advanced 1: depth × gain scale map") # Title the heatmap.
plt.xlabel("typical local gain") # Label gain axis.
plt.ylabel("depth") # Label depth axis.
plt.show() # Display the heatmap.

▶ What you'll see: the region below gain 1 turns blue with depth, while the region above gain 1 turns red.

👀 Takeaway: depth magnifies tiny deviations from neutral gain into orders-of-magnitude differences.

### Advanced 2 — Backpropagate through a tanh network by hand

**Goal.** Store activations and derivatives for a small tanh network, because manual backprop exposes exactly where gradients shrink.

In [ ]:
rng_a2 = np.random.default_rng(12) # Create reproducible weights.
width_a2 = 6 # Define hidden width.
depth_a2 = 8 # Define network depth.
W_list_a2 = [0.7 * rng_a2.normal(size=(width_a2, width_a2)) / np.sqrt(width_a2) for _ in range(depth_a2)] # Initialize moderate tanh weights.
h_a2 = rng_a2.normal(size=width_a2) # Define one input vector.
activations_a2 = [h_a2] # Store activations for backprop.
derivs_a2 = [] # Store tanh derivatives.
for W_a2 in W_list_a2: # Forward pass through all layers.
    z_a2 = W_a2 @ activations_a2[-1] # Compute preactivation.
    h_next_a2 = np.tanh(z_a2) # Apply tanh.
    activations_a2.append(h_next_a2) # Store activation.
    derivs_a2.append(1 - h_next_a2 ** 2) # Store local tanh derivative.
g_a2 = np.ones(width_a2) / np.sqrt(width_a2) # Define unit-norm output gradient.
norms_a2 = [float(np.linalg.norm(g_a2))] # Track gradient norms backward.
for layer_a2 in range(depth_a2 - 1, -1, -1): # Backpropagate from last layer to first.
    g_a2 = W_list_a2[layer_a2].T @ (derivs_a2[layer_a2] * g_a2) # Apply tanh derivative then matrix transpose.
    norms_a2.append(float(np.linalg.norm(g_a2))) # Store norm after this layer.
print("backward norms:", np.round(norms_a2, 4)) # Inspect gradient flow.
assert norms_a2[-1] < norms_a2[0] # Verify shrinkage in this seeded demo.
plt.figure(figsize=(5, 3)) # Create a gradient-flow plot.
plt.plot(np.arange(len(norms_a2)), norms_a2, marker="o", color="purple") # Plot norm after each backward step.
plt.title("Advanced 2: manual tanh backprop norms") # Title the plot.
plt.xlabel("backward step") # Label steps.
plt.ylabel("gradient norm") # Label norm.
plt.show() # Display the plot.

▶ What you'll see: the gradient norm generally shrinks as it moves through tanh derivatives and weight matrices.

👀 Takeaway: manual backprop makes the gradient product visible as alternating activation derivatives and matrix transposes.

### Advanced 3 — Residual connection as a gradient shortcut

**Goal.** Compare a plain chain with a residual-style chain, because adding an identity path gives gradients a route that is not only the repeated small transform.

In [ ]:
depth_a3 = 20 # Define chain length.
alpha_a3 = 0.05 # Define a small residual transform strength.
plain_gain_a3 = alpha_a3 ** depth_a3 # Gradient through a plain repeated small transform.
residual_gain_a3 = (1 + alpha_a3) ** depth_a3 # Gradient through repeated h + alpha h blocks.
print("plain gain:", plain_gain_a3) # Inspect plain product.
print("residual gain:", round(residual_gain_a3, 3)) # Inspect residual product.
assert plain_gain_a3 < 1e-20 and round(residual_gain_a3, 3) == 2.653 # Verify the contrast.
curves_a3 = np.vstack([alpha_a3 ** np.arange(1, depth_a3 + 1), (1 + alpha_a3) ** np.arange(1, depth_a3 + 1)]) # Build both curves.
plt.figure(figsize=(5, 3)) # Create a comparison plot.
plt.plot(curves_a3[0], label="plain alpha^depth", color="crimson") # Plot plain product.
plt.plot(curves_a3[1], label="residual (1+alpha)^depth", color="teal") # Plot residual product.
plt.yscale("log") # Use log scale for huge contrast.
plt.title("Advanced 3: residual shortcut intuition") # Title the plot.
plt.xlabel("depth") # Label depth.
plt.ylabel("gradient multiplier") # Label multiplier.
plt.legend() # Show labels.
plt.show() # Display the plot.

▶ What you'll see: the plain product vanishes almost instantly, while the residual path keeps a strong route backward.

👀 Takeaway: residual connections fight vanishing gradients by adding an identity term to the derivative path.

### Advanced 4 — Clipping an exploded update for diagnosis

**Goal.** Cap a large gradient norm without changing direction, because clipping is a common safety response when products explode.

In [ ]:
g_a4 = np.array([30.0, -40.0, 120.0]) # Define an exploded gradient direction.
clip_a4 = 10.0 # Define a maximum allowed norm.
norm_a4 = float(np.linalg.norm(g_a4)) # Compute raw gradient norm.
scale_a4 = min(1.0, clip_a4 / norm_a4) # Compute clipping multiplier.
g_clip_a4 = g_a4 * scale_a4 # Apply norm clipping.
cos_a4 = float(np.dot(g_a4, g_clip_a4) / (np.linalg.norm(g_a4) * np.linalg.norm(g_clip_a4))) # Check direction preservation.
print("raw norm:", round(norm_a4, 3), "scale:", round(scale_a4, 4), "clipped norm:", round(float(np.linalg.norm(g_clip_a4)), 3)) # Inspect clipping.
print("cosine direction match:", round(cos_a4, 3)) # Inspect direction preservation.
assert round(float(np.linalg.norm(g_clip_a4)), 3) == 10.0 and round(cos_a4, 3) == 1.0 # Verify clipped norm and direction.
plt.figure(figsize=(4.5, 3)) # Create a norm comparison plot.
plt.bar(["raw", "clipped"], [norm_a4, np.linalg.norm(g_clip_a4)], color=["crimson", "teal"]) # Compare norms.
plt.axhline(clip_a4, color="black", linestyle="--", label="clip threshold") # Mark threshold.
plt.title("Advanced 4: norm clipping") # Title the plot.
plt.ylabel("gradient norm") # Label norm axis.
plt.legend() # Show threshold label.
plt.show() # Display the plot.

▶ What you'll see: the norm falls from 130 to 10 while the cosine with the original direction stays 1.

👀 Takeaway: clipping does not fix why gradients exploded, but it can prevent one update from destroying training.

### Advanced 5 — Train a tiny deep linear model with stable and unstable scales

**Goal.** Simulate gradient descent on a deep scalar linear model, because the same product controls both prediction and parameter gradients.

In [ ]:
x_a5 = 1.0 # Define one input.
y_a5 = 1.0 # Define one target.
depth_a5 = 8 # Define number of scalar layers.
eta_a5 = 0.01 # Define learning rate.
inits_a5 = [0.7, 1.0, 1.3] # Compare shrinking, neutral, and growing initial weights.
loss_curves_a5 = [] # Store loss curves.
for init_a5 in inits_a5: # Train one scalar deep model per initialization.
    w_a5 = np.full(depth_a5, init_a5, dtype=float) # Initialize all layer weights equally.
    losses_a5 = [] # Store this run's losses.
    for step_a5 in range(60): # Run gradient descent steps.
        pred_a5 = x_a5 * np.prod(w_a5) # Forward pass through scalar product.
        loss_a5 = 0.5 * (pred_a5 - y_a5) ** 2 # Squared error loss.
        losses_a5.append(float(loss_a5)) # Save loss.
        for i_a5 in range(depth_a5): # Compute gradient for each scalar weight.
            grad_i_a5 = (pred_a5 - y_a5) * x_a5 * np.prod(np.delete(w_a5, i_a5)) # dL/dw_i for product model.
            w_a5[i_a5] -= eta_a5 * grad_i_a5 # Apply a gradient step.
        w_a5 = np.clip(w_a5, -3, 3) # Keep the educational demo finite.
    loss_curves_a5.append(losses_a5) # Store the run.
print("initial losses:", [round(c[0], 4) for c in loss_curves_a5]) # Inspect initial fit.
print("final losses:", [round(c[-1], 4) for c in loss_curves_a5]) # Inspect final fit.
assert loss_curves_a5[1][-1] < 1e-8 # Verify the neutral initialization is already optimal.
plt.figure(figsize=(5, 3)) # Create a training-curve plot.
for curve_a5, init_a5 in zip(loss_curves_a5, inits_a5): # Plot each initialization.
    plt.plot(curve_a5, label=f"init={init_a5}") # Draw loss curve.
plt.yscale("log") # Use log scale for loss.
plt.title("Advanced 5: deep scalar training scale") # Title the plot.
plt.xlabel("step") # Label optimization step.
plt.ylabel("loss, log scale") # Label loss.
plt.legend() # Show initialization labels.
plt.show() # Display the plot.

▶ What you'll see: the neutral initialization is stable, while too-small and too-large products start with very different gradient regimes.

👀 Takeaway: even a toy deep product shows why scale, depth, and learning rate must be designed together.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Deep gradients are products of many local factors, so numbers below one vanish and numbers above one explode.

A gradient through depth is a product of local gains. Small gains shrink the signal; large gains amplify it until updates become unstable. Save a copy to Drive to edit.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(6)


def clf_digits_ladder():
    """D1 XOR -> D2 blobs -> D3 noisy moons -> D4 digits -> D5 noisy digits."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, standardize, predict, and return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def one_hot(y, k):
    out = np.zeros((len(y), k))
    out[np.arange(len(y)), y.astype(int)] = 1.0
    return out


def softmax(z):
    shifted = z - z.max(axis=1, keepdims=True)
    exp_z = np.exp(shifted)
    return exp_z / exp_z.sum(axis=1, keepdims=True)


def random_relu_features(X, seed=0, width=24):
    rng = np.random.default_rng(seed + X.shape[1])
    W = rng.normal(0.0, 1.0 / np.sqrt(max(1, X.shape[1])), size=(X.shape[1], width))
    b = rng.normal(0.0, 0.15, size=width)
    H = np.maximum(0.0, X @ W + b)
    pair = X[:, :1] * X[:, 1:2] if X.shape[1] >= 2 else X
    return np.hstack([X, X * X, pair, H])


def batch_norm_fit(H, eps=1e-5):
    mu = H.mean(axis=0, keepdims=True)
    var = H.var(axis=0, keepdims=True)
    Z = (H - mu) / np.sqrt(var + eps)
    return Z, (mu, var, eps)


def batch_norm_apply(H, params):
    mu, var, eps = params
    return (H - mu) / np.sqrt(var + eps)


def layer_norm(H, eps=1e-5):
    mu = H.mean(axis=1, keepdims=True)
    var = H.var(axis=1, keepdims=True)
    return (H - mu) / np.sqrt(var + eps)


def group_norm(H, groups=4, eps=1e-5):
    usable = (H.shape[1] // groups) * groups
    head = H[:, :usable].reshape(H.shape[0], groups, -1)
    mu = head.mean(axis=2, keepdims=True)
    var = head.var(axis=2, keepdims=True)
    normed = ((head - mu) / np.sqrt(var + eps)).reshape(H.shape[0], usable)
    if usable == H.shape[1]:
        return normed
    return np.hstack([normed, H[:, usable:]])


def instance_norm(H, eps=1e-5):
    usable = (H.shape[1] // 8) * 8
    head = H[:, :usable].reshape(H.shape[0], 8, -1)
    mu = head.mean(axis=2, keepdims=True)
    var = head.var(axis=2, keepdims=True)
    normed = ((head - mu) / np.sqrt(var + eps)).reshape(H.shape[0], usable)
    if usable == H.shape[1]:
        return normed
    return np.hstack([normed, H[:, usable:]])


def deep_random_features(X, depth=4, scale=1.0, residual=False, seed=0):
    H = random_relu_features(X, seed=seed, width=20)
    rng = np.random.default_rng(seed + 100 + H.shape[1])
    for _ in range(depth):
        W = rng.normal(0.0, scale / np.sqrt(H.shape[1]), size=(H.shape[1], H.shape[1]))
        F = np.maximum(0.0, H @ W)
        if residual:
            H = H + 0.35 * F
        else:
            H = F
    return H


def transform_pair(x_tr, x_te, mode="plain", seed=0, scale=1.0, residual=False):
    Htr = random_relu_features(x_tr, seed=seed)
    Hte = random_relu_features(x_te, seed=seed)
    if mode == "batchnorm":
        Htr, params = batch_norm_fit(Htr)
        Hte = batch_norm_apply(Hte, params)
    if mode == "test_batchnorm_wrong":
        Htr, params = batch_norm_fit(Htr)
        Hte, _ = batch_norm_fit(Hte)
    if mode == "layernorm":
        Htr = layer_norm(Htr)
        Hte = layer_norm(Hte)
    if mode == "groupnorm":
        Htr = group_norm(Htr)
        Hte = group_norm(Hte)
    if mode == "instancenorm":
        Htr = instance_norm(Htr)
        Hte = instance_norm(Hte)
    if mode == "deep":
        Htr = deep_random_features(x_tr, depth=5, scale=scale, residual=residual, seed=seed)
        Hte = deep_random_features(x_te, depth=5, scale=scale, residual=residual, seed=seed)
    return Htr, Hte


def train_softmax_classifier(x_tr, y_tr, x_te, epsilon=0.0, epochs=40, lr=0.2, clip=None, schedule="constant", transform="plain", seed=0, scale=1.0, residual=False):
    Htr, Hte = transform_pair(x_tr, x_te, mode=transform, seed=seed, scale=scale, residual=residual)
    if len(y_tr) > 700:
        rng_sub = np.random.default_rng(seed + 700)
        idx_sub = rng_sub.choice(len(y_tr), size=700, replace=False)
        Htr = Htr[idx_sub]
        y_tr = y_tr[idx_sub]
    k = int(y_tr.max()) + 1
    Y = one_hot(y_tr, k)
    targets = (1.0 - epsilon) * Y + epsilon / k
    rng = np.random.default_rng(seed + 10)
    W = rng.normal(0.0, 0.01, size=(Htr.shape[1], k))
    b = np.zeros(k)
    losses = []
    grad_norms = []
    for epoch in range(epochs):
        eta = lr_value(schedule, epoch, epochs, lr)
        P = softmax(Htr @ W + b)
        loss = -np.mean(np.sum(targets * np.log(P + 1e-12), axis=1))
        G = (P - targets) / len(y_tr)
        dW = Htr.T @ G
        db = G.sum(axis=0)
        norm = float(np.sqrt(np.sum(dW * dW) + np.sum(db * db)))
        if clip is not None:
            factor = min(1.0, clip / (norm + 1e-12))
            dW = dW * factor
            db = db * factor
        W = W - eta * dW
        b = b - eta * db
        losses.append(float(loss))
        grad_norms.append(norm)
    preds = np.argmax(Hte @ W + b, axis=1)
    return preds, losses, grad_norms


def lr_value(schedule, epoch, epochs, base):
    if schedule == "constant":
        return base
    if schedule == "step":
        return base if epoch < epochs // 2 else base * 0.2
    if schedule == "cosine":
        return 0.02 * base + 0.5 * (base - 0.02 * base) * (1.0 + math.cos(math.pi * epoch / max(1, epochs - 1)))
    if schedule == "warmup_cosine":
        warm = max(2, epochs // 5)
        if epoch < warm:
            return base * (epoch + 1) / warm
        span = max(1, epochs - warm - 1)
        t = epoch - warm
        return 0.02 * base + 0.5 * (base - 0.02 * base) * (1.0 + math.cos(math.pi * t / span))
    if schedule == "onecycle":
        half = max(1, epochs // 2)
        if epoch < half:
            return base * (0.2 + 1.8 * epoch / half)
        return base * (2.0 - 1.8 * (epoch - half) / max(1, epochs - half))
    return base


def component_accuracy(name, X, y, **kwargs):
    def build(x_tr, y_tr, x_te):
        preds, _, _ = train_softmax_classifier(x_tr, y_tr, x_te, **kwargs)
        return preds
    return clf_accuracy(build, X, y)


def fit_softmax_on_features(Htr, y_tr, Hte, epochs=40, lr=0.3, seed=0):
    if len(y_tr) > 700:
        rng_sub = np.random.default_rng(seed + 701)
        idx_sub = rng_sub.choice(len(y_tr), size=700, replace=False)
        Htr = Htr[idx_sub]
        y_tr = y_tr[idx_sub]
    k = int(y_tr.max()) + 1
    Y = one_hot(y_tr, k)
    rng = np.random.default_rng(seed + Htr.shape[1])
    W = rng.normal(0.0, 0.01, size=(Htr.shape[1], k))
    b = np.zeros(k)
    for epoch in range(epochs):
        P = softmax(Htr @ W + b)
        G = (P - Y) / len(y_tr)
        dW = Htr.T @ G
        db = G.sum(axis=0)
        W = W - lr * dW
        b = b - lr * db
    return np.argmax(Hte @ W + b, axis=1)


def logistic_accuracy_for_features(X, y, mode="plain", scale=1.0, residual=False, seed=0):
    def build(x_tr, y_tr, x_te):
        Htr, Hte = transform_pair(x_tr, x_te, mode=mode, seed=seed, scale=scale, residual=residual)
        return fit_softmax_on_features(Htr, y_tr, Hte, epochs=40, lr=0.35, seed=seed)
    return clf_accuracy(build, X, y)


def ladder_preview(rungs):
    for name, X, y in rungs:
        classes = np.unique(y)
        print(f"{name:36s} X={X.shape} classes={len(classes)} sample_y={classes[:5].tolist()}")
    print("First D1 sample:", rungs[0][1][0].tolist(), "label=", int(rungs[0][2][0]))


def print_metric_table(rows, header="rung metric"):
    print(header)
    for name, metric in rows:
        print(f"{name:36s} {metric:.3f}")


def split_for_demo(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def plot_ladder_results(rungs, metrics, title, artifact_fn=None):
    fig, axes = plt.subplots(1, 5, figsize=(16, 3))
    for ax, (name, X, y) in zip(axes, rungs):
        if artifact_fn is None:
            if X.shape[1] == 64:
                ax.imshow(X[0].reshape(8, 8), cmap="gray")
            else:
                ax.scatter(X[:, 0], X[:, 1], c=y, cmap="tab10", s=12)
        else:
            artifact_fn(ax, name, X, y)
        ax.set_title(name.split()[0])
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle(title + " artifacts")
    plt.show()

    plt.figure(figsize=(6, 3))
    plt.plot(range(1, 6), metrics, marker="o")
    plt.xticks(range(1, 6), ["D1", "D2", "D3", "D4", "D5"])
    plt.ylim(0.0, 1.05)
    plt.ylabel("held-out accuracy")
    plt.title(title + " summary")
    plt.grid(True, alpha=0.3)
    plt.show()

## The concept, built once

The lesson formula is $\left\|\frac{\partial L}{\partial h_0}\right\|\approx\left(\prod_{\ell=1}^{L}s_\ell\right)\left\|\frac{\partial L}{\partial h_L}\right\|$. With layer scales $[0.5,0.8,1.2]$ and final gradient norm $2.0$, the product is $0.48$ and the backpropagated norm is $0.96$.

In [ ]:
scales = np.array([0.5, 0.8, 1.2])
product = float(np.prod(scales))
back_norm = product * 2.0
print("scale product:", product)
print("backpropagated norm:", back_norm)
assert np.isclose(product, 0.48)
assert np.isclose(back_norm, 0.96)

This helper is the reusable method for the rest of the notebook. It keeps the model and ladder fixed, then varies only this topic's component.

In [ ]:
print('Reusable component method is available in the setup cell and verified above.')

## The dataset ladder

We use the shared F5 classification ladder: D1 XOR, D2 blobs, D3 noisy moons, D4 real sklearn digits, and D5 noisy digits. The same accuracy wrapper and model family run on every rung.

In [ ]:
rungs = clf_digits_ladder()
ladder_preview(rungs)

## Run the same method across D1–D5

The table reports one held-out accuracy per rung while the component-specific sweep is printed for auditability.

In [ ]:
rungs = clf_digits_ladder()
rows = []
for rung_id, (name, X, y) in enumerate(rungs):
    vals = []
    for scale in [0.45, 0.85, 1.45]:
        acc = logistic_accuracy_for_features(X, y, mode="deep", scale=scale, residual=False, seed=60 + rung_id)
        vals.append((scale, acc))
    chosen = [acc for scale, acc in vals if scale == 0.85][0]
    rows.append((name, chosen))
    print(name, [(scale, round(acc, 3)) for scale, acc in vals])
metrics = [metric for _, metric in rows]
print_metric_table(rows, "stable-scale accuracy")

## Results visualization

The closing figure has two parts: a small multiple showing each rung's data artifact and a summary curve of the selected metric from D1 to D5.

In [ ]:
plot_ladder_results(rungs, metrics, '6.18 Vanishing and exploding gradients')

## Pitfall on D5

Ignoring scale makes D5 gradients vanish or explode. Fixes include He-style scaling, normalization, residuals, or clipping depending on which failure is observed.

In [ ]:
for scale in [0.45, 0.85, 1.45]:
    gain_product = scale ** 8
    print("depth-8 gain product at scale", scale, "=", round(gain_product, 6))
name, X, y = clf_digits_ladder()[-1]
vanish = logistic_accuracy_for_features(X, y, mode="deep", scale=0.45, residual=False, seed=96)
stable = logistic_accuracy_for_features(X, y, mode="deep", scale=0.85, residual=True, seed=96)
print("D5 vanishing plain accuracy:", round(vanish, 3))
print("D5 residual stable accuracy:", round(stable, 3))

## Evaluate it

- Metric: held-out accuracy from `clf_accuracy`; compare to a no-skill majority-class or plain-feature baseline.
- Sanity check: D1 should be inspectable and every probability target should sum to one when probabilities are used.
- Ablation: turn this topic's component off and verify the metric or diagnostic changes.
- Failure signal: unstable loss, axis mismatch, train/eval leakage, or D5 improvement without a matching diagnostic.

## Practice

1. Change one component value and rerun the D1 assertion plus the D1–D5 table.

2. Add a majority-class baseline to the summary curve.

3. On D5, print one extra diagnostic that would catch the named pitfall.